<a href="https://colab.research.google.com/github/fralfaro/ICS40125/blob/main/docs/labs/lab_03.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# ICS40125 - Laboratorio N°03





**Objetivo**: Aplicar técnicas avanzadas de manipulación y análisis de datos con pandas sobre un conjunto real de datos de contenido de Netflix, reforzando buenas prácticas y métodos eficientes sin recurrir a `groupby`, `merge`, `pivot`, ni `join`.



**Dataset**:

Trabajaremos con el archivo `netflix_titles.csv`, que contiene información sobre los títulos disponibles en la plataforma Netflix hasta el año 2021.

| Variable       | Clase     | Descripción                                                                 |
|----------------|-----------|------------------------------------------------------------------------------|
| show_id        | caracter  | Identificador único del título en el catálogo de Netflix.                   |
| type           | caracter  | Tipo de contenido: 'Movie' o 'TV Show'.                                     |
| title          | caracter  | Título del contenido.                                                       |
| director       | caracter  | Nombre del director (puede ser nulo).                                       |
| cast           | caracter  | Lista de actores principales (puede ser nulo).                              |
| country        | caracter  | País o países donde se produjo el contenido.                                |
| date_added     | fecha     | Fecha en la que el título fue agregado al catálogo de Netflix.              |
| release_year   | entero    | Año de lanzamiento original del título.                                     |
| rating         | caracter  | Clasificación por edad (por ejemplo: 'PG-13', 'TV-MA').                      |
| duration       | caracter  | Duración del contenido (minutos o número de temporadas para series).        |
| listed_in      | caracter  | Categorías o géneros en los que está clasificado el contenido.              |
| description    | caracter  | Breve sinopsis del contenido.                                               |




In [5]:
import pandas as pd

# Cargar datos
df = pd.read_csv('https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



### Parte 1: Limpieza y preparación

1. Revisar y describir el dataset:

   * ¿Cuántas filas y columnas tiene?
   * ¿Qué tipos de datos hay?
   * ¿Cuántos valores nulos hay por columna?

2. Transformar la columna `date_added` a tipo fecha.

3. Crear columnas auxiliares con `assign`:

   * Año (`year_added`)
   * Mes (`month_added`)



In [19]:
print("Número de filas y columnas:", df.shape)
print("\nTipos de datos:\n")
print(df.dtypes)
print("\nValores nulos por columna:\n")
print(df.isnull().sum())



Número de filas y columnas: (8807, 16)

Tipos de datos:

show_id                     object
type                        object
title                       object
director                    object
cast                        object
country                     object
date_added          datetime64[ns]
release_year                 int64
rating                      object
duration                    object
listed_in                   object
description                 object
year_added                 float64
month_added                float64
duration_minutes           float64
is_long_content             object
dtype: object

Valores nulos por columna:

show_id                0
type                   0
title                  0
director            2634
cast                 825
country              831
date_added            10
release_year           0
rating                 4
duration               3
listed_in              0
description            0
year_added            10
month_added    

In [20]:
import pandas as pd
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', errors='coerce')
print("Primeras 5 filas de 'date_added' con el nuevo formato:")
print(df['date_added'].head())

Primeras 5 filas de 'date_added' con el nuevo formato:
0   2021-09-25
1   2021-09-24
2   2021-09-24
3   2021-09-24
4   2021-09-24
Name: date_added, dtype: datetime64[ns]


In [21]:
df = df.assign(
    year_added=df['date_added'].dt.year,
    month_added=df['date_added'].dt.month
)
print("Columnas 'year_added' y 'month_added' creadas:")
print(df[['date_added', 'year_added', 'month_added']].head())

Columnas 'year_added' y 'month_added' creadas:
  date_added  year_added  month_added
0 2021-09-25      2021.0          9.0
1 2021-09-24      2021.0          9.0
2 2021-09-24      2021.0          9.0
3 2021-09-24      2021.0          9.0
4 2021-09-24      2021.0          9.0


## Parte 2: Técnicas avanzadas de pandas

4. Utilizar `.loc` para seleccionar películas (`type == 'Movie'`) que fueron agregadas después del año 2018.

5. Utilizar `str.contains()` y `str.extract()`:

   * Filtrar títulos que contienen la palabra 'love' (sin distinguir mayúsculas/minúsculas).
   * Extraer la duración en minutos para las películas desde la columna `duration`.

6. Aplicar `explode()` sobre la columna `listed_in` para obtener una fila por cada género.

7. Obtener un top 10 de géneros más frecuentes utilizando `value_counts()`.

8. Aplicar `where()` y `mask()` para marcar las películas de más de 120 minutos como contenido largo en una nueva columna.

9. Utilizar `.loc` para filtrar películas que cumplen con:

   * Más de 100 minutos de duración.
   * Rating igual a `'R'`.
   * País igual a `'United States'`.

10. Utilizar `.style` para formatear visualmente el top 10 de películas más largas.

In [24]:
# 4. Seleccionar películas que fueron agregadas después del año 2018.
movies_after_2018 = df.loc[(df['type'] == 'Movie') & (df['year_added'] > 2018)]

# Títulos de estas primeras 5 películas
print("\nTítulos de estas primeras 5 películas añadidas después de 2018:\n")
print(movies_after_2018.head()['title'])

print("\nPelículas añadidas después de 2018 (primeras 5 filas completas):\n")
print(movies_after_2018.head())


Títulos de estas primeras 5 películas añadidas después de 2018:

0                 Dick Johnson Is Dead
6     My Little Pony: A New Generation
7                              Sankofa
9                         The Starling
12                        Je Suis Karl
Name: title, dtype: object

Películas añadidas después de 2018 (primeras 5 filas completas):

   show_id   type                             title  \
0       s1  Movie              Dick Johnson Is Dead   
6       s7  Movie  My Little Pony: A New Generation   
7       s8  Movie                           Sankofa   
9      s10  Movie                      The Starling   
12     s13  Movie                      Je Suis Karl   

                         director  \
0                 Kirsten Johnson   
6   Robert Cullen, José Luis Ucha   
7                    Haile Gerima   
9                  Theodore Melfi   
12            Christian Schwochow   

                                                 cast  \
0                                 

In [26]:
# 5. Filtrar títulos que contienen la palabra 'love'
love_titles = df.loc[df['title'].str.contains('love', case=False, na=False)]

# Extraer la duración en minutos para las películas desde la columna duration.
movie_durations = df.loc[df['type'] == 'Movie', 'duration'].str.extract(r'(\d+)\smin')
df.loc[df['type'] == 'Movie', 'duration_minutes'] = movie_durations[0].astype(float)

# Filtrar películas que contienen 'love' y mostrar solo el título y la duración.
movies_with_love_and_duration = df.loc[
    (df['title'].str.contains('love', case=False, na=False)) & (df['type'] == 'Movie')
]
print(
    "\nTítulos de películas que contienen 'love' y su duración (primeras 5 filas):\n"
)
print(
    movies_with_love_and_duration[['title', 'duration_minutes']].head()
)


Títulos de películas que contienen 'love' y su duración (primeras 5 filas):

                         title  duration_minutes
158    Love Don't Cost a Thing             101.0
159             Love in a Puff             103.0
206  LSD: Love, Sex Aur Dhokha             112.0
227                Really Love              95.0
246                Man in Love             115.0


In [33]:
# 6. Aplicar explode() sobre la columna listed_in para obtener una fila por cada género.
df_exploded = df.assign(listed_in=df['listed_in'].str.split(', ')).explode('listed_in')

print("Título|Género")

# Obtener el título de una película por cada género
one_movie_per_genre = df_exploded.groupby('listed_in').first().reset_index()
for index, row in one_movie_per_genre.iterrows():
    print(f"{row['title']}|{row['listed_in']}")

Título|Género
The Stronghold|Action & Adventure
InuYasha the Movie 2: The Castle Beyond the Looking Glass|Anime Features
Yowamushi Pedal|Anime Series
The Great British Baking Show|British TV Shows
My Little Pony: A New Generation|Children & Family Movies
The Walking Dead|Classic & Cult TV
Jaws|Classic Movies
The Starling|Comedies
Ganglands|Crime TV Shows
Blade Runner: The Final Cut|Cult Movies
Dick Johnson Is Dead|Documentaries
Jailbirds New Orleans|Docuseries
Sankofa|Dramas
Same Kind of Different as Me|Faith & Spirituality
Dark Skies|Horror Movies
Sankofa|Independent Movies
Sankofa|International Movies
Blood & Water|International TV Shows
Tayo and Little Wizards|Kids' TV
Titipo Titipo|Korean TV Shows
Snervous Tyler Oakley|LGBTQ Movies
American Masters: Inventing David Geffen|Movies
Minsara Kanavu|Music & Musicals
Jailbirds New Orleans|Reality TV
Jeans|Romantic Movies
Kota Factory|Romantic TV Shows
Dark Skies|Sci-Fi & Fantasy
Countdown: Inspiration4 Mission to Space|Science & Nature TV

In [35]:
# 7. Obtener un top 10 de géneros más frecuentes
top_10_genres = df_exploded['listed_in'].value_counts().head(10)
print("\nTop 10 géneros más frecuentes:\n")
print(top_10_genres)


Top 10 géneros más frecuentes:

listed_in
International Movies        2752
Dramas                      2427
Comedies                    1674
International TV Shows      1351
Documentaries                869
Action & Adventure           859
TV Dramas                    763
Independent Movies           756
Children & Family Movies     641
Romantic Movies              616
Name: count, dtype: int64


In [41]:
# 8. Aplicar where() y mask() para marcar las películas de más de 120 minutos como contenido largo.
df['is_long_content'] = 'No'
df['is_long_content'] = df['is_long_content'].mask(df['duration_minutes'] > 120, 'Sí')

print("DataFrame con la nueva columna 'is_long_content' (primeras 5 filas con duración de minutos disponible):")
print(df[['title', 'duration_minutes', 'is_long_content']].dropna(subset=['duration_minutes']).head())

DataFrame con la nueva columna 'is_long_content' (primeras 5 filas con duración de minutos disponible):
                               title  duration_minutes is_long_content
0               Dick Johnson Is Dead              90.0              No
6   My Little Pony: A New Generation              91.0              No
7                            Sankofa             125.0              Sí
9                       The Starling             104.0              No
12                      Je Suis Karl             127.0              Sí


In [42]:
# 9. Utilizar .loc para filtrar películas que cumplen con:
#    * Más de 100 minutos de duración.
#    * Rating igual a 'R'.
#    * País igual a 'United States'.
filtered_movies_loc = df.loc[
    (df['duration_minutes'] > 100) &
    (df['rating'] == 'R') &
    (df['country'] == 'United States')
]

print("Películas que cumplen los criterios (duración > 100 min, rating 'R', país 'United States'):")
print(filtered_movies_loc[['title', 'duration_minutes', 'rating', 'country']].head())

Películas que cumplen los criterios (duración > 100 min, rating 'R', país 'United States'):
                           title  duration_minutes rating        country
48                  Training Day             122.0      R  United States
81                          Kate             106.0      R  United States
131  Blade Runner: The Final Cut             117.0      R  United States
139           Do the Right Thing             120.0      R  United States
144                  House Party             104.0      R  United States


In [44]:
# 10. Utilizar .style para formatear visualmente el top 10 de películas más largas.
top_10_longest_movies = df[df['type'] == 'Movie'].sort_values(by='duration_minutes', ascending=False).head(10)

print("Top 10 películas más largas con formato visual:")
# Aplicar estilo para resaltar la columna de duración
display(top_10_longest_movies[['title', 'duration_minutes', 'rating']].style.background_gradient(cmap='Blues', subset=['duration_minutes']))

Top 10 películas más largas con formato visual:


,title,duration_minutes,rating
4253,Black Mirror: Bandersnatch,312.000000,TV-MA
717,Headspace: Unwind Your Mind,273.000000,TV-G
2491,The School of Mischief,253.000000,TV-14
2487,No Longer kids,237.000000,TV-14
2484,Lock Your Girls In,233.000000,TV-PG
2488,Raya and Sakina,230.000000,TV-14
166,Once Upon a Time in America,229.000000,R
7932,Sangam,228.000000,TV-14
1019,Lagaan,224.000000,PG
4573,Jodhaa Akbar,214.000000,TV-14




### Pregunta Desafío

11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
    (Sugerencia: utilizar `value_counts` con `subset=["genre", "rating"]` después de aplicar `explode()`).



### Bonus: Análisis de duplicados y limpieza

12. ¿Existen películas con el mismo nombre (`title`) pero con distinto año de lanzamiento (`release_year`)?
13. ¿Cuántos títulos únicos hay en total en la columna `title`?





In [51]:
# 11. ¿Cuáles son las combinaciones más frecuentes de género y rating en el dataset?
# Utilizar value_counts con subset=['listed_in', 'rating'] después de aplicar explode().

genre_rating_combinations = df_exploded.value_counts(subset=['listed_in', 'rating'])

print("Combinaciones más frecuentes de género y rating:")
print(genre_rating_combinations.head(10))

Combinaciones más frecuentes de género y rating:
listed_in               rating
International Movies    TV-MA     1130
                        TV-14     1065
Dramas                  TV-MA      830
International TV Shows  TV-MA      714
Dramas                  TV-14      693
International TV Shows  TV-14      472
Comedies                TV-14      465
TV Dramas               TV-MA      434
Comedies                TV-MA      431
Dramas                  R          375
Name: count, dtype: int64


In [52]:
# 12. ¿Existen películas con el mismo nombre (title) pero con distinto año de lanzamiento (release_year)?
duplicate_titles_by_year = df.groupby('title')['release_year'].nunique()
movies_with_different_years = duplicate_titles_by_year[duplicate_titles_by_year > 1]

if not movies_with_different_years.empty:
    print("Películas con el mismo título pero diferente año de lanzamiento:")
    print(movies_with_different_years)
    print("\nDetalles de algunas de estas películas:")
    for title in movies_with_different_years.head(3).index:
        print(f"Título: {title}")
        print(df[df['title'] == title][['title', 'release_year', 'type', 'duration', 'date_added']])
        print("---")
else:
    print("No se encontraron películas con el mismo título pero diferente año de lanzamiento.")

No se encontraron películas con el mismo título pero diferente año de lanzamiento.


In [53]:
# 13. ¿Cuántos títulos únicos hay en total en la columna `title`?
unique_titles_count = df['title'].nunique()
print(f"El número total de títulos únicos en el dataset es: {unique_titles_count}")

El número total de títulos únicos en el dataset es: 8807
